# CIFAR-10 Preprocessing
- Subset: 10K train + 2K test
- Bicubic Upscale: 32×32 → 224×224
- Denoising → CLAHE → Sharpening
- Data Augmentation: 10K → 15K

In [1]:
%pip install scipy numpy matplotlib pillow scikit-learn opencv-python

Note: you may need to restart the kernel to use updated packages.


## 1. Load Dataset (10K subset)

In [ ]:
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import os

# Run from project root so all paths are uniform
if os.path.basename(os.getcwd()) == 'Preprocessing':
    os.chdir('..')

DATA_DIR = 'Dataset'

meta = scipy.io.loadmat(os.path.join(DATA_DIR, 'batches.meta.mat'))
label_names = [name[0] for name in meta['label_names'].flatten()]
print('Classes:', label_names)

batch = scipy.io.loadmat(os.path.join(DATA_DIR, 'data_batch_1.mat'))
train_data = batch['data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.uint8)
train_labels = batch['labels'].flatten()

test_batch = scipy.io.loadmat(os.path.join(DATA_DIR, 'test_batch.mat'))
test_data = test_batch['data'][:2000].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.uint8)
test_labels = test_batch['labels'].flatten()[:2000]

print(f'Train: {train_data.shape} | Test: {test_data.shape}')

## 2. Bicubic Upscale: 32×32 → 224×224

In [5]:
TARGET_SIZE = (224, 224)

def bicubic_upscale(images, target_size=TARGET_SIZE):
    upscaled = np.zeros((len(images), target_size[0], target_size[1], 3), dtype=np.uint8)
    for i in range(len(images)):
        img = Image.fromarray(images[i])
        upscaled[i] = np.array(img.resize(target_size, Image.BICUBIC))
        if (i + 1) % 2000 == 0:
            print(f'  Processed {i + 1}/{len(images)}')
    return upscaled

print('Upscaling training images to 224x224...')
train_upscaled = bicubic_upscale(train_data)
print(f'Done! Shape: {train_upscaled.shape}')

print('\nUpscaling test images to 224x224...')
test_upscaled = bicubic_upscale(test_data)
print(f'Done! Shape: {test_upscaled.shape}')

Upscaling training images to 224x224...
  Processed 2000/10000
  Processed 4000/10000
  Processed 6000/10000
  Processed 8000/10000
  Processed 10000/10000
Done! Shape: (10000, 224, 224, 3)

Upscaling test images to 224x224...
  Processed 2000/2000
Done! Shape: (2000, 224, 224, 3)


## 3. Denoising → CLAHE → Sharpening

In [6]:
def preprocess_image(img):
    """Apply: Gaussian Denoising -> CLAHE -> Unsharp Masking"""
    
    # Step 1: Gaussian Denoising (remove noise first)
    denoised = cv2.GaussianBlur(img, (3, 3), 0.5)
    
    # Step 2: CLAHE on each channel (enhance contrast)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    channels = cv2.split(denoised)
    clahe_channels = [clahe.apply(ch) for ch in channels]
    clahe_img = cv2.merge(clahe_channels)
    
    # Step 3: Unsharp Masking (sharpen edges last)
    blurred = cv2.GaussianBlur(clahe_img, (3, 3), 1.0)
    sharpened = cv2.addWeighted(clahe_img, 1.5, blurred, -0.5, 0)
    
    return sharpened

def preprocess_batch(images, name=''):
    processed = np.zeros_like(images)
    for i in range(len(images)):
        processed[i] = preprocess_image(images[i])
        if (i + 1) % 2000 == 0:
            print(f'  {name} Processed {i + 1}/{len(images)}')
    return processed

print('Applying Denoise + CLAHE + Sharpen on training images...')
train_processed = preprocess_batch(train_upscaled, 'Train:')
print('Applying Denoise + CLAHE + Sharpen on test images...')
test_processed = preprocess_batch(test_upscaled, 'Test:')
print('Done!')

Applying Denoise + CLAHE + Sharpen on training images...
  Train: Processed 2000/10000
  Train: Processed 4000/10000
  Train: Processed 6000/10000
  Train: Processed 8000/10000
  Train: Processed 10000/10000
Applying Denoise + CLAHE + Sharpen on test images...
  Test: Processed 2000/2000
Done!


## 4. Before vs After Comparison

In [ ]:
np.random.seed(42)
sample_idx = np.random.choice(len(train_data), 8, replace=False)

fig, axes = plt.subplots(3, 8, figsize=(20, 8))
fig.suptitle('Original (32×32) → Upscaled (224×224) → Processed', fontsize=13, fontweight='bold')
for i, idx in enumerate(sample_idx):
    axes[0, i].imshow(train_data[idx]); axes[0, i].axis('off')
    axes[0, i].set_title(f'{label_names[train_labels[idx]]}\n32×32', fontsize=8)
    axes[1, i].imshow(train_upscaled[idx]); axes[1, i].axis('off')
    axes[1, i].set_title('224×224', fontsize=8)
    axes[2, i].imshow(train_processed[idx]); axes[2, i].axis('off')
    axes[2, i].set_title('Processed', fontsize=8)
plt.tight_layout()
plt.savefig('Figure/before_after_comparison.png', dpi=150)
plt.show()

## 5. Data Augmentation (10K → 15K)

In [8]:
def cutout(img, size=32):
    """Randomly mask out a square patch"""
    h, w = img.shape[:2]
    y = np.random.randint(0, h)
    x = np.random.randint(0, w)
    y1 = max(0, y - size // 2)
    y2 = min(h, y + size // 2)
    x1 = max(0, x - size // 2)
    x2 = min(w, x + size // 2)
    result = img.copy()
    result[y1:y2, x1:x2] = 0
    return result

def augment_image(img):
    aug = img.copy()
    
    # Random horizontal flip (50%)
    if np.random.random() > 0.5:
        aug = np.fliplr(aug)
    
    # Random rotation (-15 to +15 degrees)
    angle = np.random.uniform(-15, 15)
    h, w = aug.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    aug = cv2.warpAffine(aug, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    
    # Random crop (pad 16px, crop back to 224x224)
    pad = 16
    padded = np.pad(aug, ((pad, pad), (pad, pad), (0, 0)), mode='reflect')
    x = np.random.randint(0, 2 * pad)
    y = np.random.randint(0, 2 * pad)
    aug = padded[y:y+224, x:x+224]
    
    # Random brightness (0.8 to 1.2)
    factor = np.random.uniform(0.8, 1.2)
    aug = np.clip(aug * factor, 0, 255).astype(np.uint8)
    
    # Cutout (50% chance)
    if np.random.random() > 0.5:
        aug = cutout(aug, size=32)
    
    return aug

# Generate 5,000 augmented images (balanced: 500 per class)
NUM_AUGMENTED = 5000
np.random.seed(42)
samples_per_class = NUM_AUGMENTED // 10

aug_images = []
aug_labels = []

for cls in range(10):
    cls_indices = np.where(train_labels == cls)[0]
    chosen = np.random.choice(cls_indices, samples_per_class, replace=True)
    for idx in chosen:
        aug_images.append(augment_image(train_processed[idx]))
        aug_labels.append(cls)
    print(f'  Augmented {label_names[cls]}: {samples_per_class} images')

aug_images = np.array(aug_images)
aug_labels = np.array(aug_labels)

# Combine original + augmented
train_final = np.concatenate([train_processed, aug_images])
labels_final = np.concatenate([train_labels, aug_labels])

# Shuffle
shuffle_idx = np.random.permutation(len(train_final))
train_final = train_final[shuffle_idx]
labels_final = labels_final[shuffle_idx]

print(f'\nOriginal:  {train_processed.shape[0]:,} images')
print(f'Augmented: {aug_images.shape[0]:,} images')
print(f'Total:     {train_final.shape[0]:,} images')
print(f'Shape:     {train_final.shape}')

  Augmented airplane: 500 images
  Augmented automobile: 500 images
  Augmented bird: 500 images
  Augmented cat: 500 images
  Augmented deer: 500 images
  Augmented dog: 500 images
  Augmented frog: 500 images
  Augmented horse: 500 images
  Augmented ship: 500 images
  Augmented truck: 500 images

Original:  10,000 images
Augmented: 5,000 images
Total:     15,000 images
Shape:     (15000, 224, 224, 3)


## 6. Show Augmented Samples

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle('Processed vs Augmented Examples', fontsize=14, fontweight='bold')
np.random.seed(99)
for i in range(8):
    idx = np.random.choice(np.where(train_labels == i)[0])
    axes[0, i].imshow(train_processed[idx]); axes[0, i].axis('off')
    axes[0, i].set_title(f'{label_names[i]}\nProcessed', fontsize=8)
    axes[1, i].imshow(augment_image(train_processed[idx])); axes[1, i].axis('off')
    axes[1, i].set_title('Augmented', fontsize=8)
plt.tight_layout()
plt.savefig('Figure/augmentation_samples.png', dpi=150)
plt.show()

## 7. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, labels, title in zip(axes, [train_labels, labels_final], ['Original (10K)', 'After Augmentation (15K)']):
    counts = np.bincount(labels)
    ax.bar(range(10), counts, color='steelblue')
    ax.set_xticks(range(10)); ax.set_xticklabels(label_names, rotation=45, ha='right')
    ax.set_ylabel('Count'); ax.set_title(title)
    for j, c in enumerate(counts):
        ax.text(j, c + 20, str(c), ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('Figure/class_distribution.png', dpi=150)
plt.show()

## 8. Normalize & Standardize

In [11]:
train_norm = train_final.astype(np.float32) / 255.0
test_norm = test_processed.astype(np.float32) / 255.0

channel_mean = train_norm.mean(axis=(0, 1, 2))
channel_std = train_norm.std(axis=(0, 1, 2))
print(f'Channel means (R,G,B): {channel_mean}')
print(f'Channel stds  (R,G,B): {channel_std}')

train_standardized = (train_norm - channel_mean) / channel_std
test_standardized = (test_norm - channel_mean) / channel_std

print(f'\nTrain - min: {train_standardized.min():.3f}, max: {train_standardized.max():.3f}, mean: {train_standardized.mean():.6f}')
print(f'Test  - min: {test_standardized.min():.3f}, max: {test_standardized.max():.3f}, mean: {test_standardized.mean():.6f}')

Channel means (R,G,B): [0.02229116 0.02229116 0.02229116]
Channel stds  (R,G,B): [0.14930223 0.14930223 0.14930223]

Train - min: -0.149, max: 6.549, mean: 3.260235
Test  - min: -0.149, max: 6.549, mean: 3.318731


## 9. One-Hot Encode Labels

In [12]:
num_classes = 10
labels_onehot = np.eye(num_classes)[labels_final]
test_labels_onehot = np.eye(num_classes)[test_labels]

print(f'Train labels one-hot: {labels_onehot.shape}')
print(f'Test labels one-hot:  {test_labels_onehot.shape}')
print(f'Example — label: {labels_final[0]} ({label_names[labels_final[0]]}) → {labels_onehot[0]}')

Train labels one-hot: (15000, 10)
Test labels one-hot:  (2000, 10)
Example — label: 6 (frog) → [0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]


## 10. Train / Validation Split

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train_int, y_val_int = train_test_split(
    train_standardized, labels_final,
    test_size=0.2, random_state=42, stratify=labels_final
)

X_test = test_standardized
y_test_int = test_labels

y_train = np.eye(num_classes)[y_train_int]
y_val = np.eye(num_classes)[y_val_int]
y_test = np.eye(num_classes)[y_test_int]

print(f'Training set:   {X_train.shape} | {y_train.shape}')
print(f'Validation set: {X_val.shape}  | {y_val.shape}')
print(f'Test set:       {X_test.shape}  | {y_test.shape}')

Training set:   (12000, 224, 224, 3) | (12000, 10)
Validation set: (3000, 224, 224, 3)  | (3000, 10)
Test set:       (2000, 224, 224, 3)  | (2000, 10)


## 11. Save Preprocessed Data

In [ ]:
out_path = 'Preprocessing/cifar10_224_preprocessed.npz'
np.savez_compressed(out_path,
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test,
    y_train_int=y_train_int, y_val_int=y_val_int, y_test_int=y_test_int,
    channel_mean=channel_mean, channel_std=channel_std,
    label_names=label_names
)
print(f'Saved: {out_path} ({os.path.getsize(out_path) / (1024**3):.2f} GB)')

## 12. PyTorch DataLoaders

In [15]:
import torch
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_train.astype(np.float32)),
        torch.from_numpy(y_train_int.astype(np.int64)),
    ),
    batch_size=BATCH_SIZE, shuffle=True
)

val_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_val.astype(np.float32)),
        torch.from_numpy(y_val_int.astype(np.int64)),
    ),
    batch_size=128
)

test_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_test.astype(np.float32)),
        torch.from_numpy(y_test_int.astype(np.int64)),
    ),
    batch_size=128
)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

Train batches: 188
Val batches:   24
Test batches:  16


## Summary

| Step | Details |
|---|---|
| Dataset | Subset: 10K + 5K augmented = **15K total** |
| Upscale | 32×32 → **224×224** (Bicubic) |
| Denoising | Gaussian blur (3×3, σ=0.5) |
| Contrast | CLAHE (clipLimit=2.0, tile=8×8) |
| Sharpening | Unsharp masking (1.5x) |
| Augmentation | Flip + Rotation + Crop + Brightness + Cutout |
| Normalize | Per-channel standardize |
| Split | 12K train / 3K val / 2K test |